# FT-Transformer for UK used-car prices
Train a small supervised transformer to estimate listing prices in pounds. Run all cells in order. Each epoch prints training loss, validation MAE and RMSE, and elapsed time. The final cell lets you change the example car.

Each numeric or categorical field becomes a learned token. Self-attention combines the tokens and a learned summary token feeds the price regressor. This compact implementation uses pre-normalized attention blocks and ReGLU feed-forward layers. [FT-Transformer paper](https://arxiv.org/abs/2106.11959) · [Dataset revision](https://github.com/MazenBassem/Used-Car-Price-Prediction-Market-Pattern-Discovery-Data-Mining/tree/afa7ea823febad0e4ebcdf0cf61a9b9cf1c1fb8e)

In [ ]:
from pathlib import Path
from datetime import datetime
from time import perf_counter
import copy
import sys

import numpy as np
import pandas as pd
import torch
from torch import nn
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

# Jupyter can start the kernel from the repository or the notebooks folder.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src.uk_modeling import CATEGORIES, NUMBERS, load_splits

output_dir = project_root / "outputs" / "uk-modeling"
output_dir.mkdir(parents=True, exist_ok=True)


SEED, EPOCHS, PATIENCE, BATCH = 42, 100, 12, 1024
torch.manual_seed(SEED)
torch.set_num_threads(4)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
def progress(message):
    line = f"{datetime.now():%H:%M:%S}  {message}"
    print(line, flush=True)
    with (output_dir / "ft_training.log").open("a", encoding="utf-8") as log:
        print(line, file=log)

plt.rcParams.update({"figure.figsize": (8, 3.8), "axes.spines.top": False,
                     "axes.spines.right": False, "figure.autolayout": True})


progress(f"Training device: {device}")

## Create three reproducible data splits
Reserve 20% of distinct vehicle profiles for testing, then 20% of the remaining profiles for validation. That gives approximately 64% / 16% / 20% of rows, matching the Random Forest experiment. **Training** fits preprocessing and network weights. **Validation** selects the best epoch. **Testing** measures the selected checkpoint once.

In [ ]:
train, valid, test = load_splits()
row_count = len(train) + len(valid) + len(test)
for name, part in [("Training", train), ("Validation", valid), ("Testing", test)]:
    progress(f"{name:10s} {len(part):,} rows ({len(part) / row_count:.1%})")

## Prepare the inputs
Build category dictionaries, numeric medians and scaling factors using training rows. Unknown categories receive a reserved token. A separate indicator records missing engine sizes. We train on log prices and report prediction errors in pounds.

In [ ]:
levels = {col: sorted(train[col].dropna().unique().tolist()) for col in CATEGORIES}
medians = train[NUMBERS].median()
filled = train[NUMBERS].fillna(medians)
means, scales = filled.mean(), filled.std().replace(0, 1)
log_center = float(np.log(train.price).mean())

def encode(frame):
    categories = [pd.Categorical(frame[col], categories=levels[col]).codes.astype("int64") + 1
                  for col in CATEGORIES]
    scaled = (frame[NUMBERS].fillna(medians) - means) / scales
    values = np.column_stack([scaled, frame.engine_size.isna()]).astype("float32")
    return (torch.tensor(values, device=device),
            torch.tensor(np.column_stack(categories), dtype=torch.long, device=device))

x_num, x_cat = encode(train)
v_num, v_cat = encode(valid)
y = torch.tensor(np.log(train.price.to_numpy()) - log_center, dtype=torch.float32, device=device)

## Define the feature tokenizer and attention network

In [ ]:
class AttentionBlock(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.norm1 = nn.LayerNorm(width)
        self.attn = nn.MultiheadAttention(width, 4, batch_first=True, dropout=0.05)
        self.norm2 = nn.LayerNorm(width)
        self.expand = nn.Linear(width, 4 * width)
        self.project = nn.Linear(2 * width, width)
        self.drop = nn.Dropout(0.05)

    def forward(self, x):
        z = self.norm1(x)
        x = x + self.drop(self.attn(z, z, z, need_weights=False)[0])
        a, gate = self.expand(self.norm2(x)).chunk(2, dim=-1)
        return x + self.drop(self.project(a * gate.relu()))

class FTTransformer(nn.Module):
    def __init__(self, numeric_count, cardinalities, width=32):
        super().__init__()
        self.weights = nn.Parameter(torch.randn(numeric_count, width) * 0.05)
        self.bias = nn.Parameter(torch.randn(numeric_count, width) * 0.05)
        self.embeddings = nn.ModuleList(nn.Embedding(count, width) for count in cardinalities)
        self.cls = nn.Parameter(torch.randn(1, 1, width) * 0.05)
        self.blocks = nn.Sequential(AttentionBlock(width), AttentionBlock(width))
        self.head = nn.Sequential(nn.LayerNorm(width), nn.ReLU(), nn.Linear(width, 1))

    def forward(self, numeric, category):
        numbers = numeric.unsqueeze(-1) * self.weights + self.bias
        labels = torch.stack([emb(category[:, i]) for i, emb in enumerate(self.embeddings)], dim=1)
        tokens = torch.cat([self.cls.expand(len(numbers), -1, -1), numbers, labels], dim=1)
        return self.head(self.blocks(tokens)[:, 0]).squeeze(-1)

def new_model():
    return FTTransformer(len(NUMBERS) + 1, [len(levels[col])+1 for col in CATEGORIES]).to(device)

@torch.inference_mode()
def price_predictions(model, numeric, category):
    model.eval()
    batches = [model(numeric[i:i+BATCH], category[i:i+BATCH]) for i in range(0, len(numeric), BATCH)]
    return torch.cat(batches).add(log_center).exp().cpu().numpy()

## Train and select a checkpoint
One epoch visits every training row. AdamW minimizes smooth L1 loss on centered log prices. Save the epoch with the lowest validation MAE in pounds and stop after twelve consecutive epochs without improvement, up to 100 epochs.

In [ ]:
model = new_model()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.0001)
loss_fn = nn.SmoothL1Loss(beta=0.1)
best_mae, best_epoch, history = float("inf"), 0, []
for epoch in range(1, EPOCHS + 1):
    start = perf_counter()
    model.train()
    total = torch.zeros((), device=device)
    for idx in torch.randperm(len(train), device=device).split(BATCH):
        optimizer.zero_grad(set_to_none=True)
        loss = loss_fn(model(x_num[idx], x_cat[idx]), y[idx])
        loss.backward()
        optimizer.step()
        total += loss.detach() * len(idx)
    pred = price_predictions(model, v_num, v_cat)
    mae = mean_absolute_error(valid.price, pred)
    rmse = root_mean_squared_error(valid.price, pred)
    train_loss = total.item() / len(train)
    history.append(dict(epoch=epoch, loss=train_loss, MAE=mae, RMSE=rmse))
    improved = mae < best_mae
    if improved:
        best_epoch, best_mae = epoch, mae
        best_weights = copy.deepcopy(model.state_dict())
    progress(f"Epoch {epoch:02d}/{EPOCHS} | train loss {train_loss:.4f} | val MAE £{mae:,.0f} | "
             f"val RMSE £{rmse:,.0f} | {perf_counter()-start:.1f}s" + (" | best" if improved else ""))
    if epoch - best_epoch >= PATIENCE:
        progress("Early stopping")
        break
model.load_state_dict(best_weights)
state = dict(weights={k: v.cpu() for k, v in best_weights.items()}, epoch=best_epoch,
             levels=levels, medians=medians.to_dict(), means=means.to_dict(),
             scales=scales.to_dict(), log_center=log_center, numeric=NUMBERS, categorical=CATEGORIES)
model_path = output_dir / "ft_price.pt"
temporary_path = model_path.with_suffix(".tmp")
torch.save(state, temporary_path)
temporary_path.replace(model_path)
progress(f"Saved epoch {best_epoch} to {model_path.name}")

In [ ]:
hist = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), layout="constrained")
axes[0].plot(hist.epoch, hist.loss, "o-", color="#245d80")
axes[0].set(title="Training loss", xlabel="Epoch", ylabel="Smooth L1 (log price)")
axes[1].plot(hist.epoch, hist.MAE, "o-", color="#217766")
axes[1].axvline(best_epoch, ls=":", color="black", label="Selected epoch")
axes[1].axhline(1167.55, ls="--", color="#a35227", label="Random Forest benchmark")
axes[1].set(title="Validation error", xlabel="Epoch", ylabel="Mean absolute error (£)")
axes[1].legend()
plt.show()

## Evaluate the selected model
The comparison uses the same held-out vehicle profiles. The original Random Forest used training and validation rows for its final refit. The transformer below retains its validation-selected weights fitted on the training partition.

In [ ]:
pred = price_predictions(model, *encode(test))
mae = mean_absolute_error(test.price, pred)
rmse = root_mean_squared_error(test.price, pred)
r2 = r2_score(test.price, pred)
progress(f"Test ({len(test):,} listings) | MAE £{mae:,.2f} | RMSE £{rmse:,.2f} | R² {r2:.3f}")
display(pd.DataFrame([[best_mae, mae, rmse, r2], [1167.55, 1138.48, 1897.52, 0.9633]],
                     index=["FT-Transformer", "Random Forest (previous run)"],
                     columns=["Validation MAE (£)", "Test MAE (£)", "Test RMSE (£)", "Test R²"]).round(3))
fig, ax = plt.subplots()
bins = ax.hexbin(test.price, pred, xscale="log", yscale="log", gridsize=45, bins="log", mincnt=1)
limits = [min(test.price.min(), pred.min()), max(test.price.max(), pred.max())]
ax.plot(limits, limits, "--", color="darkorange", label="Exact prediction")
ax.set(title="Predicted and listed prices on the test set", xlabel="Listed price (£, log scale)",
       ylabel="Predicted price (£, log scale)")
ax.legend()
fig.colorbar(bins, ax=ax, label="Listings per bin")
plt.show()

## Predict a car price
Change the details and run this cell again. Mileage is in miles, engine size in litres, annual tax in pounds and fuel economy in UK mpg.

In [ ]:
car = {"brand": "Ford", "model": "Fiesta", "year": 2018, "mileage": 35000,
       "transmission": "Manual", "fuel_type": "Petrol", "engine_size": 1.0, "tax": 150, "mpg": 60.1}
estimate = price_predictions(model, *encode(pd.DataFrame([car])))[0]
print(f"{car['year']} {car['brand']} {car['model']} | estimated price £{estimate:,.0f}")